# 🌊 Smart Water Health Monitoring & Early Warning System
## Data Preprocessing, Model Training & Comparison

**Project:** 7th Semester Minor Project  
**Two ML Tasks:**
1. **Water Quality Classification** → Low / Medium / High (12 water chemistry features)
2. **Waterborne Disease Prediction** → 8 disease classes (26 features incl. water + environmental + symptoms)

**10 Classification Models** trained and compared per task:
Logistic Regression, Decision Tree, Random Forest, Gradient Boosting, XGBoost, KNN, Linear SVM, Naive Bayes, AdaBoost, Extra Trees

## 1. Imports & Setup

In [5]:
import pandas as pd
import numpy as np
import joblib
import os
import time
import json
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier, GradientBoostingClassifier,
    AdaBoostClassifier, ExtraTreesClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Paths
MODELS_DIR = "models"
CHARTS_DIR = "charts"
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(CHARTS_DIR, exist_ok=True)

print("All imports loaded ✓")

ModuleNotFoundError: No module named 'pandas'

## 2. Data Loading & Exploration

In [ ]:
# Load dataset
df = pd.read_csv("waterborne_disease_dataset.csv")
print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumns: {list(df.columns)}")
print(f"\nData Types:\n{df.dtypes}")
print(f"\nMissing values: {df.isnull().sum().sum()}")

In [ ]:
# Target distribution
print("Disease Distribution:")
print(df["disease"].value_counts())
print(f"\nWater Quality Index Stats:")
print(df["water_quality_index"].describe())

In [ ]:
# Quick EDA visualizations
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# WQI distribution
axes[0, 0].hist(df["water_quality_index"], bins=50, color="#2196F3", edgecolor="white", alpha=0.8)
axes[0, 0].axvline(x=40, color="red", linestyle="--", label="Low/Medium boundary (40)")
axes[0, 0].axvline(x=70, color="green", linestyle="--", label="Medium/High boundary (70)")
axes[0, 0].set_title("Water Quality Index Distribution", fontweight="bold")
axes[0, 0].set_xlabel("WQI"); axes[0, 0].legend()

# Disease counts
disease_counts = df["disease"].value_counts()
axes[0, 1].barh(disease_counts.index, disease_counts.values, color="#4CAF50", edgecolor="white")
axes[0, 1].set_title("Disease Distribution", fontweight="bold")
axes[0, 1].set_xlabel("Count")

# pH distribution
axes[1, 0].hist(df["ph"], bins=50, color="#FF9800", edgecolor="white", alpha=0.8)
axes[1, 0].set_title("pH Distribution", fontweight="bold")
axes[1, 0].set_xlabel("pH")

# Turbidity distribution
axes[1, 1].hist(df["turbidity_ntu"], bins=50, color="#F44336", edgecolor="white", alpha=0.8)
axes[1, 1].set_title("Turbidity Distribution", fontweight="bold")
axes[1, 1].set_xlabel("Turbidity (NTU)")

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "0_eda_distributions.png"), dpi=150, bbox_inches="tight")
plt.show()
print("EDA visualizations saved ✓")

## 3. Data Preprocessing

### Key Design Decision: Feature Selection
- **Water Quality Model:** Uses ONLY 12 water chemistry features (no state/district)
  - This ensures the model responds to ANY change in water parameters
- **Disease Model:** Uses 26 features (water + environmental + symptoms + WQI)
  - WQI is included as a direct water quality indicator
- **State/District REMOVED** from features — they were dominating predictions and making the model insensitive to actual water parameter changes

In [ ]:
# Define feature sets
WQ_FEATURES = [
    "water_source_encoded", "water_treatment_encoded",
    "ph", "turbidity_ntu", "dissolved_oxygen_mg_l", "bod_mg_l",
    "fecal_coliform_per_100ml", "total_coliform_per_100ml",
    "tds_mg_l", "nitrate_mg_l", "fluoride_mg_l", "arsenic_ug_l",
]

DISEASE_FEATURES = [
    "water_source_encoded", "water_treatment_encoded",
    "water_quality_index",
    "ph", "turbidity_ntu", "dissolved_oxygen_mg_l", "bod_mg_l",
    "fecal_coliform_per_100ml", "total_coliform_per_100ml",
    "tds_mg_l", "nitrate_mg_l", "fluoride_mg_l", "arsenic_ug_l",
    "open_defecation_rate",
    "avg_temperature_c", "avg_rainfall_mm", "avg_humidity_pct",
    "flooding",
    "symptom_diarrhea", "symptom_vomiting", "symptom_fever",
    "symptom_abdominal_pain", "symptom_dehydration",
    "symptom_jaundice", "symptom_bloody_stool", "symptom_skin_rash",
]

print(f"Water Quality features: {len(WQ_FEATURES)}")
print(f"Disease features: {len(DISEASE_FEATURES)}")

In [ ]:
# Create Water Quality Labels from WQI
df["water_quality_label"] = pd.cut(
    df["water_quality_index"],
    bins=[0, 40, 70, 100],
    labels=["Low", "Medium", "High"],
    include_lowest=True
)
print("Water Quality Label Distribution:")
print(df["water_quality_label"].value_counts())

In [ ]:
# Encode categorical variables
le_source = LabelEncoder()
df["water_source_encoded"] = le_source.fit_transform(df["water_source"])

le_treatment = LabelEncoder()
df["water_treatment_encoded"] = le_treatment.fit_transform(df["water_treatment"])

le_wq = LabelEncoder()
le_wq.fit(["Low", "Medium", "High"])
df["water_quality_encoded"] = le_wq.transform(df["water_quality_label"])

le_disease = LabelEncoder()
df["disease_encoded"] = le_disease.fit_transform(df["disease"])

# Save all encoders
encoders = {
    "water_source": le_source,
    "water_treatment": le_treatment,
    "water_quality_label": le_wq,
    "disease": le_disease,
}
joblib.dump(encoders, os.path.join(MODELS_DIR, "label_encoders.pkl"))

print(f"Water Sources: {list(le_source.classes_)}")
print(f"Treatments: {list(le_treatment.classes_)}")
print(f"WQ Labels: {list(le_wq.classes_)}")
print(f"Diseases: {list(le_disease.classes_)}")
print("\nEncoders saved to models/label_encoders.pkl ✓")

In [ ]:
# Sample 200K rows for training efficiency
SAMPLE_SIZE = 200_000
df_sample = df.sample(SAMPLE_SIZE, random_state=42).reset_index(drop=True)
print(f"Sampled {df_sample.shape[0]:,} rows for training")

## 4. Model Training Setup

In [ ]:
def get_models():
    return {
        "Logistic Regression": LogisticRegression(max_iter=1000, n_jobs=-1, random_state=42),
        "Decision Tree": DecisionTreeClassifier(max_depth=20, random_state=42),
        "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42),
        "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
        "XGBoost": XGBClassifier(n_estimators=100, max_depth=6, use_label_encoder=False,
                                 eval_metric="mlogloss", n_jobs=-1, random_state=42, verbosity=0),
        "K-Nearest Neighbors": KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
        "Linear SVM": LinearSVC(max_iter=2000, random_state=42),
        "Naive Bayes": GaussianNB(),
        "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42, algorithm="SAMME"),
        "Extra Trees": ExtraTreesClassifier(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42),
    }

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "precision": round(precision_score(y_test, y_pred, average="weighted", zero_division=0), 4),
        "recall": round(recall_score(y_test, y_pred, average="weighted", zero_division=0), 4),
        "f1_score": round(f1_score(y_test, y_pred, average="weighted", zero_division=0), 4),
    }

def train_all(task_name, X_train, X_test, y_train, y_test, label_names):
    print(f"\n{'─'*60}")
    print(f"  TASK: {task_name}")
    print(f"  Features: {X_train.shape[1]}  |  Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}")
    print(f"{'─'*60}")

    models = get_models()
    results = {}
    best_score, best_model, best_name = 0, None, ""

    for name, model in models.items():
        t0 = time.time()
        print(f"  Training {name} ... ", end="", flush=True)
        try:
            model.fit(X_train, y_train)
            m = evaluate(model, X_test, y_test)
            m["training_time_sec"] = round(time.time() - t0, 1)
            results[name] = m
            print(f"Acc={m['accuracy']:.4f}  F1={m['f1_score']:.4f}  ({m['training_time_sec']}s)")
            if m["f1_score"] > best_score:
                best_score, best_model, best_name = m["f1_score"], model, name
        except Exception as e:
            print(f"FAILED: {e}")
            results[name] = {"error": str(e)}

    print(f"\n  ★ Best: {best_name} (F1={best_score:.4f})")
    y_pred_best = best_model.predict(X_test)
    print(classification_report(y_test, y_pred_best, target_names=label_names))
    cm = confusion_matrix(y_test, y_pred_best)
    return results, best_model, best_name, cm

print("Training functions defined ✓")

## 5. Task 1 — Water Quality Classification (Low / Medium / High)

Using **12 water chemistry features only** — no state/district encoding.  
This makes the model genuinely sensitive to changes in any individual water parameter.

In [ ]:
# Prepare features and scale
X_wq = df_sample[WQ_FEATURES].values
y_wq = df_sample["water_quality_encoded"].values

wq_scaler = StandardScaler()
X_wq_scaled = wq_scaler.fit_transform(X_wq)
joblib.dump(wq_scaler, os.path.join(MODELS_DIR, "wq_scaler.pkl"))

X_tr, X_te, y_tr, y_te = train_test_split(X_wq_scaled, y_wq, test_size=0.2, random_state=42, stratify=y_wq)
wq_labels = list(le_wq.classes_)

# Train all 10 models
wq_results, best_wq_model, best_wq_name, wq_cm = train_all(
    "Water Quality Classification (12 features)", X_tr, X_te, y_tr, y_te, wq_labels
)

# Save best model
joblib.dump(best_wq_model, os.path.join(MODELS_DIR, "best_water_quality_model.pkl"))
print(f"\nBest WQ model saved: {best_wq_name} ✓")

## 6. Task 2 — Waterborne Disease Prediction (8 Classes)

Using **26 features**: water quality parameters + environmental conditions + symptoms + WQI.  
Diseases: Cholera, Dysentery, Giardiasis, Hepatitis A, Hepatitis E, Leptospirosis, No Disease, Typhoid

In [ ]:
# Prepare features and scale
X_disease = df_sample[DISEASE_FEATURES].values
y_disease = df_sample["disease_encoded"].values

disease_scaler = StandardScaler()
X_disease_scaled = disease_scaler.fit_transform(X_disease)
joblib.dump(disease_scaler, os.path.join(MODELS_DIR, "disease_scaler.pkl"))

X_tr, X_te, y_tr, y_te = train_test_split(X_disease_scaled, y_disease, test_size=0.2, random_state=42, stratify=y_disease)
disease_labels = list(le_disease.classes_)

# Train all 10 models
disease_results, best_disease_model, best_disease_name, disease_cm = train_all(
    "Disease Prediction (26 features)", X_tr, X_te, y_tr, y_te, disease_labels
)

# Save best model
joblib.dump(best_disease_model, os.path.join(MODELS_DIR, "best_disease_model.pkl"))
print(f"\nBest Disease model saved: {best_disease_name} ✓")

## 7. Save Training Results

In [ ]:
all_results = {
    "water_quality_classification": {"best_model": best_wq_name, "results": wq_results},
    "disease_prediction": {"best_model": best_disease_name, "results": disease_results},
    "wq_features": WQ_FEATURES,
    "disease_features": DISEASE_FEATURES,
    "sample_size": SAMPLE_SIZE,
}
with open(os.path.join(MODELS_DIR, "training_results.json"), "w") as f:
    json.dump(all_results, f, indent=2)
print("Training results saved to models/training_results.json ✓")

## 8. Model Comparison — Numerical Tables

In [ ]:
def show_comparison(results, best_name, title):
    print(f"\n{'='*94}")
    print(f"  {title}")
    print(f"{'='*94}")
    print(f"{'Rank':<5} | {'Model':<24} | {'Accuracy':<10} | {'Precision':<10} | {'Recall':<10} | {'F1-Score':<10} | {'Time':<8}")
    print("-" * 94)
    sorted_m = sorted(
        [(n, m) for n, m in results.items() if "error" not in m],
        key=lambda x: x[1]["f1_score"], reverse=True
    )
    for rank, (name, m) in enumerate(sorted_m, 1):
        tag = " ★" if name == best_name else ""
        print(f" #{rank:<3} | {name + tag:<24} | {m['accuracy']*100:>8.2f}% | "
              f"{m['precision']*100:>8.2f}% | {m['recall']*100:>8.2f}% | "
              f"{m['f1_score']*100:>8.2f}% | {m['training_time_sec']:>6.1f}s")
    print(f"{'='*94}")

show_comparison(wq_results, best_wq_name, "WATER QUALITY CLASSIFICATION (Low / Medium / High)")
show_comparison(disease_results, best_disease_name, "WATERBORNE DISEASE PREDICTION (8 Classes)")

## 9. Model Comparison — Visual Charts

In [ ]:
COLORS = ["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0",
          "#00BCD4", "#795548", "#607D8B", "#E91E63", "#3F51B5"]
GOLD = "#FFD600"

def to_df(results):
    rows = [{"Model": n, "Accuracy": m["accuracy"]*100, "Precision": m["precision"]*100,
             "Recall": m["recall"]*100, "F1 Score": m["f1_score"]*100, "Time (s)": m["training_time_sec"]}
            for n, m in results.items() if "error" not in m]
    return pd.DataFrame(rows).sort_values("F1 Score", ascending=True).reset_index(drop=True)

wq_df = to_df(wq_results)
ds_df = to_df(disease_results)
metrics = ["Accuracy", "Precision", "Recall", "F1 Score"]

### 9.1 Grouped Bar Charts — All Metrics

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(24, 8))

for ax, df, best, title in [
    (axes[0], wq_df, best_wq_name, "Water Quality"),
    (axes[1], ds_df, best_disease_name, "Disease Prediction"),
]:
    x = np.arange(len(df)); w = 0.18
    for i, met in enumerate(metrics):
        bars = ax.bar(x + i*w, df[met], w, label=met, color=COLORS[i], edgecolor="white")
        for bar, model in zip(bars, df["Model"]):
            if model == best:
                bar.set_edgecolor(GOLD); bar.set_linewidth(2.5)
    ax.set_xticks(x + 1.5*w); ax.set_xticklabels(df["Model"], rotation=35, ha="right", fontsize=9)
    ax.set_ylim(70, 100); ax.set_ylabel("Score (%)", fontweight="bold")
    ax.set_title(f"{title} — All Metrics", fontsize=13, fontweight="bold")
    ax.legend(loc="lower right"); ax.grid(axis="y", alpha=0.3)
    ax.annotate(f"★ Best: {best}", xy=(0.01, 0.95), xycoords="axes fraction", fontsize=10,
                fontweight="bold", color="#1B5E20",
                bbox=dict(boxstyle="round,pad=0.3", facecolor="#C8E6C9", edgecolor="#1B5E20"))

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "1_all_metrics_comparison.png"), dpi=150, bbox_inches="tight")
plt.show()

### 9.2 F1 Score Ranking

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, df, best, title in [
    (axes[0], wq_df, best_wq_name, "Water Quality — F1 Ranking"),
    (axes[1], ds_df, best_disease_name, "Disease Prediction — F1 Ranking"),
]:
    colors = [GOLD if m == best else COLORS[0] for m in df["Model"]]
    bars = ax.barh(df["Model"], df["F1 Score"], color=colors, edgecolor="white", height=0.6)
    for bar, val in zip(bars, df["F1 Score"]):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f"{val:.2f}%", va="center", fontsize=9, fontweight="bold")
    ax.set_xlabel("F1 Score (%)", fontweight="bold")
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlim(70, 100); ax.grid(axis="x", alpha=0.3)

plt.suptitle("F1 Score Ranking — Side by Side", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "3_f1_score_ranking_both.png"), dpi=150, bbox_inches="tight")
plt.show()

### 9.3 Confusion Matrices — Best Models

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, cm, lbls, cmap, best, title in [
    (axes[0], wq_cm, wq_labels, "Blues", best_wq_name, "Water Quality"),
    (axes[1], disease_cm, disease_labels, "Oranges", best_disease_name, "Disease Prediction"),
]:
    im = ax.imshow(cm, cmap=cmap, aspect="auto")
    ax.set_xticks(range(len(lbls))); ax.set_yticks(range(len(lbls)))
    rot = 45 if len(lbls) > 4 else 0
    ax.set_xticklabels(lbls, rotation=rot, ha="right" if rot else "center", fontsize=8 if len(lbls)>4 else 10)
    ax.set_yticklabels(lbls, fontsize=8 if len(lbls)>4 else 10)
    ax.set_xlabel("Predicted", fontweight="bold"); ax.set_ylabel("Actual", fontweight="bold")
    ax.set_title(f"{title} — {best}\nConfusion Matrix", fontsize=12, fontweight="bold")
    for i in range(len(lbls)):
        for j in range(len(lbls)):
            color = "white" if cm[i, j] > cm.max()/2 else "black"
            ax.text(j, i, f"{cm[i,j]:,}", ha="center", va="center", color=color,
                    fontsize=7 if len(lbls)>4 else 11, fontweight="bold")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "6_confusion_matrices.png"), dpi=150, bbox_inches="tight")
plt.show()

### 9.4 Radar Chart — Top 5 Disease Models

In [ ]:
top5 = ds_df.nlargest(5, "F1 Score")
cats = ["Accuracy", "Precision", "Recall", "F1 Score"]
angles = np.linspace(0, 2*np.pi, len(cats), endpoint=False).tolist() + [0]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
for idx, (_, row) in enumerate(top5.iterrows()):
    vals = [row[c] for c in cats] + [row[cats[0]]]
    ax.plot(angles, vals, "o-", linewidth=2, label=row["Model"], color=COLORS[idx])
    ax.fill(angles, vals, alpha=0.08, color=COLORS[idx])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(cats, fontsize=11, fontweight="bold")
ax.set_ylim(80, 100)
ax.set_title("Top 5 Disease Models — Radar Comparison", fontsize=13, fontweight="bold", pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "5_radar_top5_disease_models.png"), dpi=150, bbox_inches="tight")
plt.show()

### 9.5 Accuracy vs Training Time Trade-off

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
for ax, df, best, title in [
    (axes[0], wq_df, best_wq_name, "Water Quality"),
    (axes[1], ds_df, best_disease_name, "Disease Prediction"),
]:
    for idx, (_, row) in enumerate(df.iterrows()):
        sz = 250 if row["Model"] == best else 120
        c = GOLD if row["Model"] == best else COLORS[idx % len(COLORS)]
        ax.scatter(row["Time (s)"], row["Accuracy"], s=sz, c=c, edgecolors="black", linewidths=1, zorder=5)
        ax.annotate(row["Model"], (row["Time (s)"], row["Accuracy"]),
                    textcoords="offset points", xytext=(8, 5), fontsize=7.5)
    ax.set_xlabel("Training Time (s, log scale)", fontweight="bold")
    ax.set_ylabel("Accuracy (%)", fontweight="bold")
    ax.set_title(f"{title} — Accuracy vs Time", fontweight="bold")
    ax.set_xscale("log"); ax.xaxis.set_major_formatter(mticker.ScalarFormatter())
    ax.grid(alpha=0.3)

plt.suptitle("Accuracy vs Training Time", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CHARTS_DIR, "7_accuracy_vs_time_scatter.png"), dpi=150, bbox_inches="tight")
plt.show()

## 10. Summary

In [ ]:
print("=" * 60)
print("  PIPELINE COMPLETE ✓")
print("=" * 60)
print(f"\n  Water Quality Best Model: {best_wq_name}")
print(f"  Disease Prediction Best Model: {best_disease_name}")
print(f"\n  Saved artifacts in models/:")
for f in os.listdir(MODELS_DIR):
    print(f"    - {f}")
print(f"\n  Charts saved in charts/:")
for f in sorted(os.listdir(CHARTS_DIR)):
    print(f"    - {f}")
print(f"\n  To start the API server:")
print(f"    python -m uvicorn app:app --host 0.0.0.0 --port 8000")
print(f"    Then open http://localhost:8000/docs")